# Cartoon plots of linear vs quadratic dynamics

In [ ]:
using Plots
using ColorSchemes
green_color = :green

In [ ]:
# Taken from analysis/tracks.jl
function expocdf(t::Array, rate::Float64; t0::Float64=0.0)
    rtt0 = clamp!(rate .* (t .- t0), 0.0, Inf)
    cdf = 1.0 .- exp.(-rtt0)
    return cdf
end

function twostepcdf(t::Array, rate::Float64; t0::Float64=0.0)
    rt = clamp!(rate .* (t .- t0), 0.0, Inf)
    cdf = 1.0 .- exp.(-rt) .* (1.0 .+ rt)
    return cdf
end

function threestepcdf(t::Array, rate::Float64; t0::Float64=0.0)
    rt = clamp!(rate .* (t .- t0), 0.0, Inf)
    cdf = 1.0 .- exp.(-rt) .* (1.0 .+ rt .* (1.0 .+ 0.5 .* rt))
end

In [ ]:
# Small 'cartoon' linear vs quadratic plots for publication panels
# Creates `panels/` and saves two PDF files (one per panel) and displays them inline
mkpath("panels")
# Time vector (coarse enough for a cartoon, fine enough for smooth lines)
t = collect(0:0.5:100)
# Plot full CDF in thin line, highlight early times with an inset
linrate = 0.05
quadrate = linrate * 2.0
threerate = quadrate * 1.5
lin = expocdf(t, linrate; t0=0.0)
quad = twostepcdf(t, quadrate; t0=0.0)
three = threestepcdf(t, threerate; t0=0.0)

fitfrac = 0.4
insetfrac = 0.05
tinset = collect(0:0.5*insetfrac:100*insetfrac)
lininset = linrate * tinset  #expocdf(tinset, linrate; t0=0.0)
quadinset = twostepcdf(tinset, quadrate; t0=0.0)
threeinset = threestepcdf(tinset, threerate; t0=0.0)

# Visual styling: small but legible fonts for labels/titles
label_font = font(12)
title_font = font(10)
ylbl = "Active cells"
lw = 2.0
plotsz = (210,160)
# Individual panels (small sizes, suitable for composite figures)
# Remove ticks/ticklabels and disable grid lines (xticks/yticks = false)
p1 = plot(t, lin,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box,
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)
plot!(p1,
    tinset, lininset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    framestyle=:box,
    bg_inside = nothing, 
    label = nothing,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
# To mark where the inset was taken from, 
# define a function that returns a Plots.Shape
# https://discourse.julialang.org/t/how-to-draw-a-rectangular-region-with-plots-jl/1719/2
rectangle(w, h, x, y) = Shape(x .+ [0,w,w,0], y .+ [0,0,h,h])
rect1 = rectangle(t[end]*insetfrac*2, insetfrac*6,0,0)

plot!(p1, rect1, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)

# Add small lines connecting the boxes
xx = maximum(rect1.x)
y1, y2 = extrema(rect1.y)
xinsbox = t[end] * (1.0 - 0.5)
yinsbox = 0.75
plot!(p1, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0)
plot!(p1, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0)

p2 = plot(t, quad,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box,
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)

plot!(p2,
    tinset, quadinset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    bg_inside = nothing, 
    label = nothing,
    framestyle=:box,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
rect2 = rectangle(t[end]*insetfrac*2, insetfrac*4,0,0)
plot!(p2, rect2, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)
y1, y2 = extrema(rect2.y)
plot!(p2, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0)
plot!(p2, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0)


# Also for a three-step model
p3 = plot(t, three,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box,
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)

plot!(p3,
    tinset, threeinset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    bg_inside = nothing, 
    label = nothing,
    framestyle=:box,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
rect3 = rectangle(t[end]*insetfrac*2, insetfrac*4,0,0)
plot!(p3, rect3, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)
y1, y2 = extrema(rect3.y)
plot!(p3, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0)
plot!(p3, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0)


# Display each panel in the notebook
display(p1)
display(p2)
display(p3)

# Save each panel as a PDF (no combined plot)
savefig(p1, "figures/final_tracks_panels/linear_cartoon.pdf")
savefig(p2, "figures/final_tracks_panels/quadratic_cartoon.pdf")
savefig(p3, "figures/final_tracks_panels/three-step_cartoon.pdf")

# Print saved file paths for quick confirmation
println("Saved: figures/final_tracks_panels/linear_cartoon.pdf, "
        *"figures/final_tracks_panels/quadratic_cartoon.pdf, "
        *"figures/final_tracks_panels/three-step_cartoon.pdf")

In [ ]:
maximum(rect1.x)